In [1]:
# Importing Libraries
import pandas as pd
import nltk
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
import string
import json
import re

In [2]:
# Donwload the necessary NLTK files
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to C:\Users\Abdul Rehman
[nltk_data]     Tahir\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Abdul Rehman
[nltk_data]     Tahir\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Abdul Rehman
[nltk_data]     Tahir\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Abdul Rehman
[nltk_data]     Tahir\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Abdul Rehman
[nltk_data]     Tahir\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [3]:
# Loading the preprocessed dataset
articlesPreprocessed_dataset = pd.read_csv('Dataset/medium_articles_preprocessed.csv')

In [4]:
# Printing the shape and head of the dataset
print(articlesPreprocessed_dataset.shape, '\n')
articlesPreprocessed_dataset.head()

(192361, 10) 



,title,text,url,authors,timestamp,tags,text_length,title_length,num_tags,num_authors
0,Mental Note Vol. 24,Photo by Josh Riemer on Unsplash\n\nMerry Chri...,https://medium.com/invisible-illness/mental-no...,['Ryan Fan'],2020-12-26 03:38:10.479000+00:00,"['Mental Health', 'Health', 'Psychology', 'Sci...",5018,19,5,1
1,Your Brain On Coronavirus,Your Brain On Coronavirus\n\nA guide to the cu...,https://medium.com/age-of-awareness/how-the-pa...,['Simon Spichak'],2020-09-23 22:10:17.126000+00:00,"['Mental Health', 'Coronavirus', 'Science', 'P...",7293,25,5,1
2,Mind Your Nose,Mind Your Nose\n\nHow smell training can chang...,https://medium.com/neodotlife/mind-your-nose-f...,[],2020-10-10 20:17:37.132000+00:00,"['Biotechnology', 'Neuroscience', 'Brain', 'We...",5730,14,5,0
3,The 4 Purposes of Dreams,Passionate about the synergy between science a...,https://medium.com/science-for-real/the-4-purp...,['Eshan Samaranayake'],2020-12-21 16:05:19.524000+00:00,"['Health', 'Neuroscience', 'Mental Health', 'P...",146,24,5,1
4,Surviving a Rod Through the Head,"You’ve heard of him, haven’t you? Phineas Gage...",https://medium.com/live-your-life-on-purpose/s...,['Rishav Sinha'],2020-02-26 00:01:01.576000+00:00,"['Brain', 'Health', 'Development', 'Psychology...",2326,32,5,1


In [5]:
# Creating the list of stopwords to be removed from the text
stop_words = set(stopwords.words('english'))
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'only',
 'or',
 'other',
 'our',
 'ours',
 'ourselves',
 'out',
 'over',
 'own',
 'r

In [6]:
# Initializing the WordNetLemmatizer and SpellChecker
lemmatizer = WordNetLemmatizer()

In [7]:
# Checking the punctuation marks
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

<hr>

In [8]:
# Function to convert NLTK POS tags to WordNet POS tags
def get_wordnet_pos(tag):
    if tag.startswith('J'):  # Adjective
        return wordnet.ADJ
    elif tag.startswith('V'):  # Verb
        return wordnet.VERB
    elif tag.startswith('N'):  # Noun
        return wordnet.NOUN
    elif tag.startswith('R'):  # Adverb
        return wordnet.ADV
    else:
        return None  # Other POS

In [9]:
# Function to preprocess the batch of text
def preprocess(texts, doc_ids):
    lexicon_batch = set()
    forward_index_batch = {}

    for text, doc_id in zip(texts, doc_ids): 
        tokens = word_tokenize(text.lower())
        tokens = [
            lemmatizer.lemmatize(word, pos=get_wordnet_pos(pos_tag([word])[0][1]) or 'n')
            for word in tokens if word.isalpha()
            and word not in stop_words 
            and word not in string.punctuation
        ]

        # Updating the lexicon and forward index
        lexicon_batch.update(tokens)
        forward_index_batch[doc_id] = tokens

    return lexicon_batch, forward_index_batch

In [10]:
# Initializing the set to store the lexicon and forward index
lexicon = set()
forward_index = {}

batch_size = 5000
doc_id_prefix = "doc"

for i in range(0, len(articlesPreprocessed_dataset), batch_size):
    # Get batch texts and document ids
    batch_texts = (articlesPreprocessed_dataset['title'][i:i+batch_size] + ' ' + 
                   articlesPreprocessed_dataset['text'][i:i+batch_size])
    doc_ids = [f"{doc_id_prefix}{j}" for j in range(i, i + len(batch_texts))]

    # Preprocessing the batch with document ids
    batch_lexicon, batch_forward_index = preprocess(batch_texts, doc_ids)

    # Updating the lexicon and forward index
    lexicon.update(batch_lexicon)
    forward_index.update(batch_forward_index)

In [11]:
# Converting the created lexicon set to a sorted list
lexicon = sorted(list(lexicon))

In [12]:
def refine_lexicon(lexicon):
    repeated_char_pattern = re.compile(r'(.)\1{2,}')  # Match three or more consecutive repeated characters
    irregular_repeat_pattern = re.compile(r'(.)\1{1,}.*\1{1,}')  # Match irregular repetitions

    refined_lexicon = [
        token for token in lexicon if 2 < len(token) <= 20 
        and not repeated_char_pattern.search(token)  # No excessive consecutive repeats
        and not irregular_repeat_pattern.search(token)  # No irregular repetitions
        and token.isalpha()
    ]

    return sorted(refined_lexicon)

In [13]:
# Refining the lexicon
refined_lexicon = refine_lexicon(lexicon)

In [14]:
# Saving lexicon to a JSON file
with open('lexicon.json', 'w') as f:
    json.dump(refined_lexicon, f)

print(f'Lexicon created with {len(refined_lexicon)} unique terms.')

# Saving forward index to a JSON file
with open('forward_index.json', 'w') as f:
    json.dump(forward_index, f)

print(f'Forward index created with {len(forward_index)} documents.')

Lexicon created with 457903 unique terms.
Forward index created with 192361 documents.


In [15]:
# Reading the created lexicon from the JSON file
with open('lexicon.json', 'r') as f:
    lexicon_file = json.load(f)

print(f'Lexicon loaded with {len(lexicon_file)} terms.')
lexicon_file[:10]

Lexicon loaded with 457903 terms.


['aab',
 'aabb',
 'aabbcc',
 'aabcdcb',
 'aabps',
 'aabs',
 'aabsolute',
 'aac',
 'aacc',
 'aace']

In [16]:
# Reading the created forward index from the JSON file
with open('forward_index.json', 'r') as f:
    forward_index_file = json.load(f)

print(f'Forward index loaded with {len(forward_index_file)} documents.')
list(forward_index_file.items())[:2]

Forward index loaded with 192361 documents.


[('doc0',
  ['mental',
   'note',
   'vol',
   'photo',
   'josh',
   'riemer',
   'unsplash',
   'merry',
   'christmas',
   'happy',
   'holiday',
   'everyone',
   'want',
   'everyone',
   'know',
   'much',
   'appreciate',
   'everyone',
   'thankful',
   'reader',
   'writer',
   'anywhere',
   'without',
   'thank',
   'bring',
   'informative',
   'vulnerable',
   'important',
   'piece',
   'destigmatize',
   'mental',
   'illness',
   'mental',
   'health',
   'without',
   'ado',
   'ten',
   'top',
   'story',
   'last',
   'week',
   'curated',
   'capacity',
   'love',
   'inspire',
   'universal',
   'capacity',
   'hate',
   'discourage',
   'irrespective',
   'gender',
   'race',
   'age',
   'religion',
   'none',
   'u',
   'exempt',
   'aggressive',
   'proclivity',
   'narcissistically',
   'disorder',
   'accordingly',
   'repress',
   'deep',
   'seat',
   'feeling',
   'inferiority',
   'inflate',
   'delusion',
   'grandeur',
   'superiority',
   'prone',
   '

<hr>